# The Standard Library Toolkit

Chapter 11 laid out the standard library as a map of modules and promised that `collections`, `itertools` and `functools` would get a proper treatment in chapter 15. This is that chapter, with `datetime` added, because every dataset you will ever open has a date column in it.

Chapter 10 already used one of these without explaining it. The Amazon sales report built its totals like this:

```python
from collections import defaultdict

revenue_by_category = defaultdict(float)
orders_by_customer = defaultdict(int)
```

Both lines are about to make sense, and the second one turns out to be a job for a different tool entirely.

Everything in this chapter ships with Python. No `pip install`, no `requirements.txt`, nothing to go wrong on someone else's machine. These four modules are where you go before you go looking for a library.

**What we will learn:**

1. `collections`: `defaultdict`, `Counter`, `namedtuple`, `deque`
2. `itertools`: `chain`, `islice`, `groupby`, `product`, `combinations`, `count`, `cycle`
3. `functools`: `partial`, and a second look at `lru_cache`
4. `datetime`: parsing, formatting, and arithmetic with `timedelta`
5. The traps in each one, including the two that catch nearly everybody

### The Data We Will Use

One small dataset runs through the whole chapter, so each tool is solving a problem you can already see.

In [1]:
ORDERS = [
    # order_id, customer, category,      date,         amount
    ("A001", "Alice", "Books",       "2024-03-01",  59.98),
    ("A002", "Bob",   "Electronics", "2024-03-01",  45.00),
    ("A003", "Alice", "Appliances",  "2024-03-02",  89.99),
    ("A004", "Carol", "Books",       "2024-03-04",  89.97),
    ("A005", "Bob",   "Electronics", "2024-03-05",  75.00),
    ("A006", "Dave",  "Appliances",  "2024-03-05",  89.99),
    ("A007", "Alice", "Books",       "2024-03-08",  24.99),
    ("A008", "Carol", "Books",       "2024-03-11", 120.00),
]

print(f"{len(ORDERS)} orders, ${sum(o[4] for o in ORDERS):,.2f} total")

8 orders, $594.92 total


---
# 1. `collections`

Four container types that do a job the built-in ones do awkwardly.

## 1.1 `defaultdict`: the `KeyError` you keep working around

Grouping is the most common thing anyone does to a list of records, and a plain dictionary makes it clumsy. The problem is that the first time you touch a key, it does not exist yet:

In [2]:
revenue = {}

try:
    revenue["Books"] += 59.98
except KeyError as e:
    print("KeyError:", e)

KeyError: 'Books'


So every grouping loop grows a guard clause:

In [3]:
revenue = {}

for order_id, customer, category, date, amount in ORDERS:
    if category not in revenue:         # the guard
        revenue[category] = 0.0
    revenue[category] += amount

print(revenue)

{'Books': 294.94, 'Electronics': 120.0, 'Appliances': 179.98}


`dict.setdefault()` shortens that to one line, and plenty of code uses it:

```python
revenue[category] = revenue.setdefault(category, 0.0) + amount
```

`defaultdict` goes further. You hand it a function that produces the starting value, and it calls that function for any key you touch that is not there yet:

In [4]:
from collections import defaultdict

revenue = defaultdict(float)            # float() is 0.0

for order_id, customer, category, date, amount in ORDERS:
    revenue[category] += amount         # no guard, no setdefault

for category, total in revenue.items():
    print(f"{category:<12} ${total:>7.2f}")

Books        $ 294.94
Electronics  $ 120.00
Appliances   $ 179.98


That is what chapter 10's sales report was doing.

The argument is a **function**, not a value. `defaultdict(float)` calls `float()` to get `0.0`; `defaultdict(list)` calls `list()` to get a fresh empty list. That second one is the version you will reach for most, because it turns grouping into two lines:

In [5]:
orders_by_customer = defaultdict(list)

for order in ORDERS:
    orders_by_customer[order[1]].append(order[0])

for customer, ids in orders_by_customer.items():
    print(f"{customer:<6} {ids}")

Alice  ['A001', 'A003', 'A007']
Bob    ['A002', 'A005']
Carol  ['A004', 'A008']
Dave   ['A006']


Note `defaultdict(list)` and not `defaultdict([])`. Passing a list would hand every key the *same* list object, which is the mutable default argument bug from chapter 6 in a new costume. Python catches it for you:

In [6]:
try:
    broken = defaultdict([])            # type: ignore (deliberately wrong)
    broken["Books"].append("A001")
except TypeError as e:
    print("TypeError:", e)

TypeError: first argument must be callable or None


### The trap: reading a missing key creates it

This is the one thing about `defaultdict` that surprises people. Looking a key up is not a read-only operation:

In [7]:
counts = defaultdict(int)
counts["Books"] += 1

print("does Electronics exist?", counts["Electronics"])   # just looking
print("the dictionary now  :", dict(counts))
print("length              :", len(counts))

does Electronics exist? 0
the dictionary now  : {'Books': 1, 'Electronics': 0}
length              : 2


One innocent lookup added a key. In a report that then loops over `counts.items()`, you get a row of zeros for every category anyone ever asked about.

Use `.get()` when you only want to look:

In [8]:
counts = defaultdict(int)
counts["Books"] += 1

print("with .get():", counts.get("Electronics", 0))
print("still just:", dict(counts))

with .get(): 0
still just: {'Books': 1}


A `defaultdict` is a real `dict` underneath, so everything from chapter 3 works on it, and `dict(d)` converts it back to a plain dictionary once you are done building it.

In [9]:
print("is it a dict?", isinstance(counts, dict))
print("plain again  :", type(dict(counts)).__name__)

is it a dict? True
plain again  : dict


## 1.2 `Counter`: counting, already sorted

Chapter 10 counted orders per customer with `defaultdict(int)`. That works, but counting is common enough to have its own container. Hand `Counter` any iterable and it counts what is in it:

In [10]:
from collections import Counter

by_customer = Counter(order[1] for order in ORDERS)

print(by_customer)
print("Alice's orders:", by_customer["Alice"])

Counter({'Alice': 3, 'Bob': 2, 'Carol': 2, 'Dave': 1})
Alice's orders: 3


The reason to prefer it over `defaultdict(int)` is `most_common()`, which is the question you actually wanted answered:

In [11]:
by_category = Counter(order[2] for order in ORDERS)

print("all categories  :", by_category.most_common())
print("top 2           :", by_category.most_common(2))
print("total items     :", by_category.total())

all categories  : [('Books', 4), ('Electronics', 2), ('Appliances', 2)]
top 2           : [('Books', 4), ('Electronics', 2)]
total items     : 8


`Counter` also counts the characters in a string or the words in a sentence, which is the first thing you do to any text dataset:

In [12]:
review = "the book was good the delivery was fast the price was fair"

words = Counter(review.split())
print(words.most_common(3))

letters = Counter("mississippi")
print(sorted(letters.items()))

[('the', 3), ('was', 3), ('book', 1)]
[('i', 4), ('m', 1), ('p', 2), ('s', 4)]


### `Counter` does not create missing keys

Here is the contrast with `defaultdict`. A missing key reads as `0`, and looking does not add anything:

In [13]:
letters = Counter("mississippi")

print("count of 'z':", letters["z"])
print("is 'z' in it?", "z" in letters)
print("length still:", len(letters))

count of 'z': 0
is 'z' in it? False
length still: 4


Counters also subtract, which answers "what changed?" in one line. Chapter 3 compared sets this way; this is the same idea with quantities attached:

In [14]:
february = Counter({"Books": 3, "Electronics": 5, "Appliances": 1})
march    = Counter({"Books": 4, "Electronics": 2, "Appliances": 2})

print("month on month:", march - february)     # only positive counts survive
print("combined      :", march + february)

month on month: Counter({'Books': 1, 'Appliances': 1})
combined      : Counter({'Books': 7, 'Electronics': 7, 'Appliances': 3})


`march - february` dropped Electronics entirely, because a `Counter` subtraction keeps only positive counts. When you want the negatives too, subtract the values yourself:

In [15]:
change = {k: march[k] - february[k] for k in set(march) | set(february)}
print(dict(sorted(change.items())))

{'Appliances': 1, 'Books': 1, 'Electronics': -3}


## 1.3 `namedtuple`: a tuple whose fields have names

Our orders are tuples, and reading them is a guessing game:

In [16]:
order = ORDERS[0]

print(order[2], order[4])       # which fields are these again?

Books 59.98


Chapter 2 said tuples are for fixed records, and this is the cost: position is the only way in. `namedtuple` builds a tuple class with names attached:

In [17]:
from collections import namedtuple

Order = namedtuple("Order", "order_id customer category date amount")

orders = [Order(*row) for row in ORDERS]
first = orders[0]

print(first)
print(first.category, first.amount)

Order(order_id='A001', customer='Alice', category='Books', date='2024-03-01', amount=59.98)
Books 59.98


The `print(first)` line is the quiet win. A plain tuple prints as a row of values you have to decode; a namedtuple prints its own field names, which makes every debugging session shorter.

It is still a tuple, so everything you learned in chapter 2 still applies. It unpacks, it indexes, and it cannot be changed:

In [18]:
order_id, customer, category, date, amount = first     # unpacking still works
print(order_id, customer)

print("index access :", first[1])
print("is a tuple   :", isinstance(first, tuple))

try:
    first.amount = 0.0                  # type: ignore (a tuple cannot be changed)
except AttributeError as e:
    print("AttributeError:", e)

A001 Alice
index access : Alice
is a tuple   : True
AttributeError: can't set attribute


Two methods you will use. `_replace()` returns a **new** record with one field changed, since the original cannot be edited, and `_asdict()` converts to a dictionary, which is what you want just before writing JSON with chapter 10's `json.dump`:

In [19]:
corrected = first._replace(amount=61.98)

print("original :", first.amount)
print("corrected:", corrected.amount)
print("as a dict:", corrected._asdict())

original : 59.98
corrected: 61.98
as a dict: {'order_id': 'A001', 'customer': 'Alice', 'category': 'Books', 'date': '2024-03-01', 'amount': 61.98}


The leading underscores look like private methods, but they are public. They are named that way so that a field called `replace` or `count` cannot collide with them.

### When a namedtuple is not enough

A namedtuple is for plain data with no behaviour. Once the record needs methods, or needs to change after it is created, chapter 8's classes are the answer. Writing that class by hand means an `__init__` that assigns five fields, and a `__repr__` so it prints readably.

`dataclasses` does that for you with a decorator, which is chapter 13's `@` syntax again:

In [20]:
from dataclasses import dataclass

@dataclass
class Shipment:
    order_id: str
    carrier: str
    days_in_transit: int = 3            # a default, like any function argument

    def is_late(self):                  # methods, which a namedtuple cannot have
        return self.days_in_transit > 5


s = Shipment("A001", "BlueDart")
print(s)                                # __repr__ written for us
print("late?", s.is_late())

s.days_in_transit = 7                   # mutable, unlike a namedtuple
print("late now?", s.is_late())

Shipment(order_id='A001', carrier='BlueDart', days_in_transit=3)
late? False
late now? True


| | `namedtuple` | `@dataclass` | plain class (ch. 8) |
|---|---|---|---|
| Can change after creation | No | Yes | Yes |
| Can have methods | No | Yes | Yes |
| Behaves as a tuple | Yes | No | No |
| You write | one line | the fields | everything |

Reach for a namedtuple when you are handling rows of data, and a dataclass when the thing has behaviour.

## 1.4 `deque`: a list that is fast at both ends

`list.append()` and `list.pop()` work on the end of a list and are fast. `list.pop(0)` works on the front, and to do it Python has to shift every remaining item one place left. On a short list nobody notices. On a long one it is the reason a script that "worked on the sample" takes minutes on the full file.

`deque` (say "deck") is built to be quick at both ends. Rather than take that on trust, time it:

In [21]:
from collections import deque
import time

N = 50_000
as_list = list(range(N))
as_deque = deque(range(N))

start = time.perf_counter()
while as_list:
    as_list.pop(0)                      # shifts everything left, every time
list_time = time.perf_counter() - start

start = time.perf_counter()
while as_deque:
    as_deque.popleft()
deque_time = time.perf_counter() - start

print(f"removing {N:,} items from the front")
print("deque was faster        :", deque_time < list_time)
print("by more than 10 times   :", list_time > deque_time * 10)

removing 50,000 items from the front
deque was faster        : True
by more than 10 times   : True


The exact numbers depend on your machine, so the cell prints the comparison instead. The gap is large and it grows with the size of the data.

`deque` takes `append`/`pop` at the right and `appendleft`/`popleft` at the left:

In [22]:
queue = deque(["job-1", "job-2", "job-3"])

queue.append("job-4")                   # arrives at the back
queue.appendleft("urgent")              # jumps the queue

print(queue)
print("next up:", queue.popleft())
print("remaining:", list(queue))

deque(['urgent', 'job-1', 'job-2', 'job-3', 'job-4'])
next up: urgent
remaining: ['job-1', 'job-2', 'job-3', 'job-4']


The feature that earns `deque` a place in data work is `maxlen`. Give it one and the container keeps only the most recent N items, dropping the oldest automatically. That is a rolling window, and writing it by hand is fiddly:

In [23]:
last_three = deque(maxlen=3)

for order in ORDERS:
    last_three.append(order[0])

print("the three most recent orders:", list(last_three))

the three most recent orders: ['A006', 'A007', 'A008']


---
# 2. `itertools`

Chapter 12 taught you to write generators. `itertools` is a drawer of ready-made ones. Everything here is lazy in exactly the sense chapter 12 meant: nothing is computed until you ask for it, and nothing large is held in memory.

## 2.1 `chain`: several sequences, one loop

Three monthly exports, one report. Without `chain` you either write three loops or build one big combined list. Let us make the files first, so this is a real problem and not a toy one.

In [24]:
import csv
import os

os.makedirs("toolkit_demo", exist_ok=True)

HEADER = ["order_id", "customer", "category", "date", "amount"]
months = {"2024_01": ORDERS[:3], "2024_02": ORDERS[3:6], "2024_03": ORDERS[6:]}

for label, rows in months.items():
    with open(f"toolkit_demo/orders_{label}.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(rows)

print("wrote", len(months), "files, each with a header row")

wrote 3 files, each with a header row


In [25]:
from contextlib import ExitStack     # chapter 14
from itertools import chain

paths = [f"toolkit_demo/orders_{label}.csv" for label in months]

with ExitStack() as stack:
    files = [stack.enter_context(open(p, newline="", encoding="utf-8")) for p in paths]

    readers = []
    for f in files:
        reader = csv.reader(f)
        next(reader)                    # step over this file's header
        readers.append(reader)

    every_row = chain.from_iterable(readers)
    total = sum(float(row[4]) for row in every_row)

print(f"total across {len(paths)} files: ${total:,.2f}")

total across 3 files: $594.92


`chain.from_iterable` takes one iterable of iterables; `chain(a, b, c)` takes them as separate arguments. Both hand back an iterator, so the three files were read one row at a time and never held together in memory:

In [26]:
print(type(chain([1, 2], [3, 4])).__name__)
print(list(chain([1, 2], [3, 4], [5])))

chain
[1, 2, 3, 4, 5]


## 2.2 `islice`: slicing something you cannot slice

Chapter 2's slicing syntax needs a sequence. A generator is not one, so this fails:

In [27]:
squares = (n * n for n in range(1_000_000))

try:
    print(squares[:5])                  # type: ignore (a generator has no slicing)
except TypeError as e:
    print("TypeError:", e)

TypeError: 'generator' object is not subscriptable


`islice` slices any iterator, taking only as much as you ask for. The remaining 999,995 squares are never computed:

In [28]:
from itertools import islice

squares = (n * n for n in range(1_000_000))
print("first five:", list(islice(squares, 5)))

squares = (n * n for n in range(1_000_000))
print("items 10 to 14:", list(islice(squares, 10, 15)))

first five: [0, 1, 4, 9, 16]
items 10 to 14: [100, 121, 144, 169, 196]


The everyday use is skipping a header, which the `chain` example above did with a bare `next(reader)`. `islice` says it in one expression, and reads only the rows you asked for:

In [29]:
with open("toolkit_demo/orders_2024_03.csv", encoding="utf-8") as f:
    for line in islice(f, 1, None):     # everything after the header
        print(line.strip())

A007,Alice,Books,2024-03-08,24.99
A008,Carol,Books,2024-03-11,120.0


One limit to know: `islice` cannot take a negative index. `islice(rows, -5)` is an error, because reaching the last five items means reading to the end, and an iterator has no end until it gets there. For the last N items, use the `deque(maxlen=5)` trick from section 1.4.

## 2.3 `groupby`: the one everybody gets wrong

`groupby` runs through an iterable and groups **consecutive** items that share a key. That word "consecutive" is the whole story, and skipping past it produces a report that looks fine and is wrong.

Our orders are in date order, not category order. Group them by category as they stand:

In [30]:
from itertools import groupby

for category, group in groupby(ORDERS, key=lambda o: o[2]):
    print(f"{category:<12} count={len(list(group))}")

Books        count=1
Electronics  count=1
Appliances   count=1
Books        count=1
Electronics  count=1
Appliances   count=1
Books        count=2


Seven groups from three categories, and Books appears three separate times. Nothing raised, nothing warned, and a report built on that output would quietly understate every category.

`groupby` needs the data sorted by the same key first:

In [31]:
by_category = sorted(ORDERS, key=lambda o: o[2])

for category, group in groupby(by_category, key=lambda o: o[2]):
    rows = list(group)
    revenue = sum(row[4] for row in rows)
    print(f"{category:<12} count={len(rows)}  ${revenue:>7.2f}")

Appliances   count=2  $ 179.98
Books        count=4  $ 294.94
Electronics  count=2  $ 120.00


Sort by the key, group by the same key. If you find yourself writing one and not the other, that is the bug.

### The second trap: the group is used up

Each `group` is an iterator that is only valid while you are on it. Move to the next group and the previous one is spent. Collecting the pairs first and looping over them later gives you empty groups:

In [32]:
pairs = list(groupby(by_category, key=lambda o: o[2]))

for category, group in pairs:
    print(f"{category:<12} {list(group)}")

Appliances   []
Books        []
Electronics  []


Three categories, three empty lists, no error. Consume each group inside the loop, as the working version above does with `rows = list(group)`.

Given both traps, a `defaultdict(list)` is often the better tool: it needs no sorting and hands you real lists.

In [33]:
grouped = defaultdict(list)
for order in ORDERS:
    grouped[order[2]].append(order)

for category, rows in sorted(grouped.items()):
    print(f"{category:<12} count={len(rows)}")

Appliances   count=2
Books        count=4
Electronics  count=2


Use `groupby` when the data arrives sorted already, which is common when it comes from a database or a sorted file, and a `defaultdict(list)` the rest of the time.

## 2.4 `product` and `combinations`: every combination, without nested loops

`product` gives you every combination of several sequences. If you have ever written three nested loops to try every setting, this is the replacement, and it is exactly how a hyperparameter grid is built:

In [34]:
from itertools import product

learning_rates = [0.01, 0.1]
depths = [3, 5, 7]

grid = list(product(learning_rates, depths))

for learning_rate, depth in grid:
    print(f"  learning_rate={learning_rate:<5} max_depth={depth}")

print("combinations to try:", len(grid))

  learning_rate=0.01  max_depth=3
  learning_rate=0.01  max_depth=5
  learning_rate=0.01  max_depth=7
  learning_rate=0.1   max_depth=3
  learning_rate=0.1   max_depth=5
  learning_rate=0.1   max_depth=7
combinations to try: 6


`combinations` picks groups from a single sequence, without repeats and without caring about order. Every pair of variants in an A/B test:

In [35]:
from itertools import combinations, permutations

variants = ["control", "blue-button", "short-form", "new-copy"]

print("pairs to compare:")
for a, b in combinations(variants, 2):
    print(f"  {a} vs {b}")

print("\ncombinations:", len(list(combinations(variants, 2))))
print("permutations:", len(list(permutations(variants, 2))))

pairs to compare:
  control vs blue-button
  control vs short-form
  control vs new-copy
  blue-button vs short-form
  blue-button vs new-copy
  short-form vs new-copy

combinations: 6
permutations: 12


The difference in that last pair of numbers is the whole distinction. `combinations` treats `("control", "blue-button")` and `("blue-button", "control")` as the same thing, so it returns 6. `permutations` counts order, so it returns 12.

These grow fast. `combinations` of 2 from 20 items is 190; of 5 from 20 it is 15,504. Check the length before you loop over one.

## 2.5 `count` and `cycle`: iterators that never end

`count` is `range` without a stop value, which makes it a tidy id generator:

In [36]:
from itertools import count, cycle

order_ids = count(1001)

print([next(order_ids) for _ in range(3)])
print("and it carries on:", next(order_ids))

[1001, 1002, 1003]
and it carries on: 1004


`cycle` repeats a sequence forever, which is round-robin assignment in one line:

In [37]:
servers = cycle(["eu-west-1", "us-east-1", "ap-south-1"])

for job in ["job-1", "job-2", "job-3", "job-4", "job-5"]:
    print(f"  {job} -> {next(servers)}")

  job-1 -> eu-west-1
  job-2 -> us-east-1
  job-3 -> ap-south-1
  job-4 -> eu-west-1
  job-5 -> us-east-1


Both are infinite, so `list(count())` never returns and neither does a `for` loop over one without a `break`. This is the one warning in this chapter that we will not demonstrate, because the cell would simply hang.

Pair them with `islice` when you want a fixed number:

In [38]:
print(list(islice(count(0, 5), 6)))          # start at 0, step 5, take 6
print(list(islice(cycle("AB"), 5)))

[0, 5, 10, 15, 20, 25]
['A', 'B', 'A', 'B', 'A']


---
# 3. `functools`

Chapter 7 imported `reduce` from here and chapter 13 used `wraps` and `lru_cache`. Two more things in this module are worth your time.

## 3.1 `partial`: freezing arguments

When you keep calling the same function with the same first argument, `partial` builds a new function with that argument already filled in:

In [39]:
from functools import partial

def tag(level, message):
    return f"[{level}] {message}"


warn = partial(tag, "WARNING")
error = partial(tag, "ERROR")

print(warn("disk almost full"))
print(error("connection refused"))

[WARNING] disk almost full
[ERROR] connection refused


Chapter 13 would have you write that as a closure, and the result is the same. `partial` is the short way to say it:

In [40]:
def make_tagger(level):                 # the chapter 13 version
    def tagger(message):
        return tag(level, message)
    return tagger


warn_closure = make_tagger("WARNING")

print(warn_closure("disk almost full"))
print("same result:", warn_closure("disk almost full") == warn("disk almost full"))

[WARNING] disk almost full
same result: True


It works on built-ins too, which is where it saves the most typing. A `lambda` would do the same job, and `partial` says it more plainly:

In [41]:
from_hex = partial(int, base=16)

print(from_hex("ff"), from_hex("1a"))

to_2dp = partial(round, ndigits=2)
print([to_2dp(o[4] * 1.18) for o in ORDERS[:4]])     # amounts with 18% tax

255 26
[70.78, 53.1, 106.19, 106.16]


## 3.2 `lru_cache`, and the two ways it bites

Chapter 13 introduced `@lru_cache` as a decorator that remembers results. Two things about it were left for this chapter, and both cause real bugs.

### Arguments have to be hashable

The cache is a dictionary keyed on the arguments, so the arguments have to work as dictionary keys. Chapter 3 covered that rule: tuples yes, lists no.

In [42]:
from functools import lru_cache

@lru_cache
def total(amounts):
    print("  (computing)")
    return sum(amounts)


print(total((59.98, 45.00, 89.99)))     # a tuple is fine
print(total((59.98, 45.00, 89.99)))     # second call is free

try:
    total([59.98, 45.00])               # a list is not
except TypeError as e:
    print("TypeError:", e)

  (computing)
194.97
194.97
TypeError: unhashable type: 'list'


### A cache goes stale

`lru_cache` assumes the function returns the same answer for the same arguments forever. When the function reads something that can change, it will keep handing back the old answer:

In [43]:
prices = {"Books": 10.0}

@lru_cache
def price_of(category):
    return prices[category]


print("first call        :", price_of("Books"))

prices["Books"] = 12.0                  # the price changes
print("after the change  :", price_of("Books"))
print("cache_info        :", price_of.cache_info())

first call        : 10.0
after the change  : 10.0
cache_info        : CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


The function returned a price that no longer exists anywhere in the program, and `cache_info()` shows why: one miss, then a hit. `cache_clear()` throws the stored answers away:

In [44]:
price_of.cache_clear()

print("after cache_clear :", price_of("Books"))
print("cache_info        :", price_of.cache_info())

after cache_clear : 12.0
cache_info        : CacheInfo(hits=0, misses=1, maxsize=128, currsize=1)


Cache pure functions: same input, same output, no reading of anything that moves. Chapter 13's Fibonacci is the perfect case. A function that reads a file, a database, or a global is the wrong case.

---
# 4. `datetime`

Our orders carry dates as strings, and a date in a string is not a date. You cannot subtract it, sort it reliably, or ask what day of the week it was.

## 4.1 The four types

| Type | Holds | Example |
|---|---|---|
| `date` | year, month, day | `date(2024, 3, 1)` |
| `time` | hour, minute, second | `time(14, 30)` |
| `datetime` | both together | `datetime(2024, 3, 1, 14, 30)` |
| `timedelta` | a length of time | `timedelta(days=7)` |

`datetime` is the one you will use most.

In [45]:
from datetime import date, time, datetime, timedelta

launch = datetime(2024, 3, 1, 14, 30)

print(launch)
print("year :", launch.year)
print("day  :", launch.day)
print("date part:", launch.date())
print("time part:", launch.time())

2024-03-01 14:30:00
year : 2024
day  : 1
date part: 2024-03-01
time part: 14:30:00


`datetime.now()` gives the current moment. This notebook does not print it, because the output would be different every time it runs, but the type is worth seeing:

In [46]:
right_now = datetime.now()

print(type(right_now).__name__)
print("has a year:", right_now.year > 2000)

datetime
has a year: True


## 4.2 Parsing: string to `datetime`

`strptime` reads a string using a format you supply. The format is a set of `%` codes describing what the string looks like:

In [47]:
order_date = datetime.strptime("2024-03-01", "%Y-%m-%d")

print(order_date)
print("weekday:", order_date.strftime("%A"))

2024-03-01 00:00:00
weekday: Friday


### Why this matters: strings sort wrong

A date written day-first sorts alphabetically, which has nothing to do with time:

In [48]:
dates = ["09/03/2024", "15/02/2024", "01/12/2023"]

print("sorted as strings:", sorted(dates))

parsed = sorted(datetime.strptime(d, "%d/%m/%Y") for d in dates)
print("sorted as dates  :", [d.strftime("%d/%m/%Y") for d in parsed])

sorted as strings: ['01/12/2023', '09/03/2024', '15/02/2024']
sorted as dates  : ['01/12/2023', '15/02/2024', '09/03/2024']


The string sort put December 2023 first, then March 2024, then February 2024. Only the parsed version is in time order.

The exception is ISO 8601, the `YYYY-MM-DD` format our orders use, where the pieces run from largest to smallest and alphabetical order happens to match date order. That is the reason to store dates that way in every file you write. Python has a dedicated parser for it, and no format string to get wrong:

In [49]:
print(sorted(o[3] for o in ORDERS)[:3])        # strings, and correctly ordered

parsed = datetime.fromisoformat("2024-03-01")
print(parsed, "|", parsed.isoformat())

['2024-03-01', '2024-03-01', '2024-03-02']
2024-03-01 00:00:00 | 2024-03-01T00:00:00


When the format does not match the string, `strptime` raises, which is a `ValueError` you can catch with chapter 9's tools:

In [50]:
try:
    datetime.strptime("01/03/2024", "%Y-%m-%d")
except ValueError as e:
    print("ValueError:", e)

ValueError: time data '01/03/2024' does not match format '%Y-%m-%d'


## 4.3 Formatting: `datetime` to string

`strftime` is `strptime` backwards: it takes a datetime and a format, and gives you a string.

| Code | Means | Example |
|---|---|---|
| `%Y` | 4-digit year | `2024` |
| `%m` | month, zero-padded | `03` |
| `%d` | day, zero-padded | `01` |
| `%B` | month name | `March` |
| `%A` | weekday name | `Friday` |
| `%H:%M` | 24-hour time | `14:30` |
| `%Y-%m-%d` | ISO date | `2024-03-01` |

In [51]:
moment = datetime(2024, 3, 1, 14, 30)

print(moment.strftime("%Y-%m-%d"))
print(moment.strftime("%d %B %Y"))
print(moment.strftime("%A, %d %b %Y at %H:%M"))

2024-03-01
01 March 2024
Friday, 01 Mar 2024 at 14:30


## 4.4 `timedelta`: date arithmetic

Subtract two datetimes and you get a `timedelta`. Add a `timedelta` to a datetime and you get another datetime. That is the whole model, and it removes every "how many days has it been" loop you might be tempted to write:

In [52]:
first = datetime.fromisoformat(ORDERS[0][3])
last = datetime.fromisoformat(ORDERS[-1][3])

span = last - first

print("first order:", first.date())
print("last order :", last.date())
print("span       :", span)
print("in days    :", span.days)
print("in hours   :", span.total_seconds() / 3600)

first order: 2024-03-01
last order : 2024-03-11
span       : 10 days, 0:00:00
in days    : 10
in hours   : 240.0


In [53]:
deadline = last + timedelta(days=30)
print("payment due:", deadline.date())

reminder = deadline - timedelta(days=7, hours=12)
print("reminder at:", reminder)

payment due: 2024-04-10
reminder at: 2024-04-02 12:00:00


Use `.days` for whole days and `.total_seconds()` for everything else. There is no `.hours` or `.months` attribute, because months are not a fixed length. A "one month later" that keeps the day number is a job for `dateutil.relativedelta`, which is a third-party package and outside this chapter.

## 4.5 Naive and aware datetimes

A datetime with no timezone attached is called **naive**. One that carries a timezone is **aware**. Mixing them raises, and the error is worth meeting here rather than at three in the morning:

In [54]:
from datetime import timezone

naive = datetime(2024, 3, 1, 9, 0)
aware = datetime(2024, 3, 1, 9, 0, tzinfo=timezone.utc)

print("naive:", naive)
print("aware:", aware)

try:
    print(aware - naive)
except TypeError as e:
    print("TypeError:", e)

naive: 2024-03-01 09:00:00
aware: 2024-03-01 09:00:00+00:00
TypeError: can't subtract offset-naive and offset-aware datetimes


Pick one and stay with it. For anything that crosses a machine boundary, which includes every API and log file you will meet in chapters 17 and 18, store times in UTC as aware datetimes:

In [55]:
stamped = datetime(2024, 3, 1, 9, 0, tzinfo=timezone.utc)

print("stored as:", stamped.isoformat())
print("read back:", datetime.fromisoformat("2024-03-01T09:00:00+00:00"))

stored as: 2024-03-01T09:00:00+00:00
read back: 2024-03-01 09:00:00+00:00


---
# 5. Putting It Together

A small report that uses all four modules on the same data: `namedtuple` for readable rows, `datetime` for the dates, `defaultdict` and `Counter` for the grouping and counting, and `groupby` over data we sorted ourselves.

In [56]:
from collections import Counter, defaultdict, namedtuple
from datetime import datetime
from itertools import groupby

Order = namedtuple("Order", "order_id customer category date amount")

orders = [Order(oid, cust, cat, datetime.fromisoformat(d), amt)
          for oid, cust, cat, d, amt in ORDERS]

print(f"{len(orders)} orders from {orders[0].date.date()} to {orders[-1].date.date()}")
print(f"total revenue: ${sum(o.amount for o in orders):,.2f}\n")

revenue = defaultdict(float)
for order in orders:
    revenue[order.category] += order.amount

print("revenue by category")
for category, total in sorted(revenue.items(), key=lambda pair: -pair[1]):
    print(f"  {category:<12} ${total:>7.2f}")

print("\nbusiest weekdays")
for day, n in Counter(o.date.strftime("%A") for o in orders).most_common(3):
    print(f"  {day:<10} {n} orders")

print("\nspend per customer")
for customer, group in groupby(sorted(orders, key=lambda o: o.customer),
                               key=lambda o: o.customer):
    rows = list(group)
    print(f"  {customer:<6} orders: {len(rows)}   ${sum(r.amount for r in rows):>7.2f}")

8 orders from 2024-03-01 to 2024-03-11
total revenue: $594.92

revenue by category
  Books        $ 294.94
  Appliances   $ 179.98
  Electronics  $ 120.00

busiest weekdays
  Friday     3 orders
  Monday     2 orders
  Tuesday    2 orders

spend per customer
  Alice  orders: 3   $ 174.96
  Bob    orders: 2   $ 120.00
  Carol  orders: 2   $ 209.97
  Dave   orders: 1   $  89.99


Every line of that is standard library. On a laptop with nothing installed but Python, it runs.

---
# 6. Common Mistakes

| Mistake | What happens | Fix |
|---|---|---|
| `defaultdict([])` instead of `defaultdict(list)` | `TypeError` on creation | Pass the function, not a value |
| Looking up a key in a `defaultdict` | The key is created, silently | Use `.get(key, default)` to read |
| `groupby` without sorting first | The same key appears in several groups | Sort by the key you group by |
| Keeping `groupby` groups for later | Every group is empty | Consume each group inside the loop |
| `list(count())` or `for x in cycle(...)` | Runs forever | Wrap in `islice`, or `break` |
| `lru_cache` on a function taking a list | `TypeError: unhashable type` | Pass a tuple |
| `lru_cache` on a function that reads changing data | Stale answers, no error | Only cache pure functions |
| Sorting dates as strings | Wrong order, no error | Parse them, or store ISO 8601 |
| Subtracting an aware datetime from a naive one | `TypeError` | Keep everything in one style, ideally UTC |

---
# 7. Summary: Your Standard Library Cheat Sheet

**`collections`**

```python
from collections import defaultdict, Counter, namedtuple, deque

totals = defaultdict(float)         # missing key becomes float() == 0.0
groups = defaultdict(list)          # missing key becomes []
counts = Counter(items)             # counts anything iterable
counts.most_common(3)               # the top three, already sorted
Row = namedtuple("Row", "a b c")    # a tuple with field names
window = deque(maxlen=100)          # keeps only the last 100 items
```

**`itertools`** (everything is lazy)

| Function | Gives you |
|---|---|
| `chain(a, b)` / `chain.from_iterable(seqs)` | several sequences as one stream |
| `islice(it, stop)` / `islice(it, start, stop)` | slicing for iterators |
| `groupby(sorted_data, key=f)` | consecutive runs sharing a key |
| `product(a, b)` | every combination across sequences |
| `combinations(seq, n)` | every group of n, order ignored |
| `permutations(seq, n)` | every group of n, order counted |
| `count(start, step)` | an endless counter |
| `cycle(seq)` | an endless repeat |

**`functools`**

```python
from functools import partial, lru_cache, reduce, wraps

usd = partial(format_money, "$")    # freeze the first argument
@lru_cache                          # remember results (hashable args, pure function)
def fib(n): ...
fib.cache_info()                    # hits, misses, size
fib.cache_clear()                   # throw the stored answers away
```

**`datetime`**

```python
from datetime import date, datetime, timedelta, timezone

datetime.strptime("2024-03-01", "%Y-%m-%d")   # string  -> datetime
datetime.fromisoformat("2024-03-01")          # the same, for ISO 8601
moment.strftime("%d %B %Y")                   # datetime -> string
moment.isoformat()                            # the format to store
later = moment + timedelta(days=30)           # arithmetic
gap = later - moment                          # a timedelta
gap.days, gap.total_seconds()                 # reading it
datetime(2024, 3, 1, tzinfo=timezone.utc)     # aware, not naive
```

**Which container**

| You want | Use |
|---|---|
| Totals or groups by key | `defaultdict(float)` / `defaultdict(list)` |
| How many of each, ranked | `Counter` |
| A readable fixed record | `namedtuple` |
| A record with methods or edits | `@dataclass`, or a class |
| Fast adds and removes at the front | `deque` |
| The last N items only | `deque(maxlen=N)` |

---

**Next:** chapter 16 covers regular expressions, and starts by redoing chapter 2's hashtag and URL-slug parsing, which we wrote by hand with `.find()` and slicing.